![Cabec%CC%A7alho_notebook.png](cabecalho_notebook.png)

# PCA - Tarefa 01: *HAR* com PCA

Vamos trabalhar com a base da demonstração feita em aula, mas vamos explorar um pouco melhor como é o desempenho da árvore variando o número de componentes principais.

In [123]:
# Importa a biblioteca pandas para manipulação de dados
import pandas as pd

# Importa a classe DecisionTreeClassifier para criar e treinar árvores de decisão
from sklearn.tree import DecisionTreeClassifier

# Importa a classe PCA para realizar Análise de Componentes Principais
from sklearn.decomposition import PCA

# Importa a função accuracy_score para calcular a acurácia do modelo
from sklearn.metrics import accuracy_score

# Importa a função cross_val_score para realizar validação cruzada
from sklearn.model_selection import cross_val_score

# Importa a classe GridSearchCV para realizar busca de hiperparâmetros
from sklearn.model_selection import GridSearchCV

# Importa a função train_test_split para dividir os dados em treino e teste
from sklearn.model_selection import train_test_split

# Importa a classe Pipeline para criar pipelines de processamento de dados e modelos
from sklearn.pipeline import Pipeline

# Importa a classe StandardScaler para normalizar os dados
from sklearn.preprocessing import StandardScaler

# Importa a classe SVC para criar e treinar modelos de Máquinas de Vetores de Suporte (SVM)
from sklearn.svm import SVC

In [124]:
# Definição dos caminhos para os arquivos de dados
filename_features = "./input/UCI HAR Dataset/features.txt"  # Arquivo com os nomes das variáveis (features)
filename_labels = "./input/UCI HAR Dataset/activity_labels.txt"  # Arquivo com os rótulos das atividades

filename_subtrain = "./input/UCI HAR Dataset/train/subject_train.txt"  # Arquivo com os IDs dos sujeitos no conjunto de treino
filename_xtrain = "./input/UCI HAR Dataset/train/X_train.txt"  # Arquivo com os dados de treino (features)
filename_ytrain = "./input/UCI HAR Dataset/train/y_train.txt"  # Arquivo com os rótulos das atividades no conjunto de treino

filename_subtest = "./input/UCI HAR Dataset/test/subject_test.txt"  # Arquivo com os IDs dos sujeitos no conjunto de teste
filename_xtest = "./input/UCI HAR Dataset/test/X_test.txt"  # Arquivo com os dados de teste (features)
filename_ytest = "./input/UCI HAR Dataset/test/y_test.txt"  # Arquivo com os rótulos das atividades no conjunto de teste

# Carregando os nomes das variáveis (features) a partir do arquivo
features = pd.read_csv(filename_features, header=None, names=['nome_var'], sep="#")  # Lê o arquivo de features
features = features.squeeze()  # Converte para uma série unidimensional

# Carregando os rótulos das atividades
labels = pd.read_csv(filename_labels, delim_whitespace=True, header=None, names=['cod_label', 'label'])  # Lê o arquivo de rótulos

# Carregando os IDs dos sujeitos no conjunto de treino
subject_train = pd.read_csv(filename_subtrain, header=None, names=['subject_id'])  # Lê os IDs dos sujeitos no treino
subject_train = subject_train.squeeze()  # Converte para uma série unidimensional

# Carregando os dados de treino (features e rótulos)
X_train = pd.read_csv(filename_xtrain, delim_whitespace=True, header=None, names=features.tolist())  # Lê os dados de treino
y_train = pd.read_csv(filename_ytrain, header=None, names=['cod_label'])  # Lê os rótulos do treino

# Carregando os IDs dos sujeitos no conjunto de teste
subject_test = pd.read_csv(filename_subtest, header=None, names=['subject_id'])  # Lê os IDs dos sujeitos no teste
subject_test = subject_test.squeeze()  # Converte para uma série unidimensional

# Carregando os dados de teste (features e rótulos)
X_test = pd.read_csv(filename_xtest, delim_whitespace=True, header=None, names=features.tolist())  # Lê os dados de teste
y_test = pd.read_csv(filename_ytest, header=None, names=['cod_label'])  # Lê os rótulos do teste

C:\Users\hfasa\AppData\Local\Temp\ipykernel_26532\1463072973.py:18: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  labels = pd.read_csv(filename_labels, delim_whitespace=True, header=None, names=['cod_label', 'label'])  # Lê o arquivo de rótulos
C:\Users\hfasa\AppData\Local\Temp\ipykernel_26532\1463072973.py:25: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_train = pd.read_csv(filename_xtrain, delim_whitespace=True, header=None, names=features.tolist())  # Lê os dados de treino
C:\Users\hfasa\AppData\Local\Temp\ipykernel_26532\1463072973.py:33: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  X_test = pd.read_csv(filename_xtest, delim_whitespace=True, header=None, names=features.tolist())  # Lê os d

## Árvore de decisão

Rode uma árvore de decisão com todas as variáveis, utilizando o ```ccp_alpha=0.001```. Avalie a acurácia nas bases de treinamento e teste. Avalie o tempo de processamento.

In [125]:
%%time

# Divide os dados de treino em treino e validação, utilizando 20% dos dados para validação
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=4500)

# Cria um classificador de árvore de decisão com um parâmetro de poda (ccp_alpha) e uma semente aleatória para reprodutibilidade
clf = DecisionTreeClassifier(random_state=4500, ccp_alpha=0.001)

# Treina o classificador com os dados de treino
clf.fit(X_train, y_train.values.ravel())

# Calcula e imprime a acurácia do modelo no conjunto de treino
print("Acurácia do modelo em treino: ", round(accuracy_score(y_train, clf.predict(X_train)), 3))

# Calcula e imprime a acurácia do modelo no conjunto de teste (validação)
print("Acurácia do modelo em teste: ", round(accuracy_score(y_test, clf.predict(X_test)), 3))

Acurácia do modelo em treino:  0.977
Acurácia do modelo em teste:  0.935
CPU times: total: 3.25 s
Wall time: 3.26 s


## Árvore com PCA

Faça uma análise de componentes principais das variáveis originais. Utilize apenas uma componente. Faça uma árvore de decisão com esta componente como variável explicativa.

- Avalie a acurácia nas bases de treinamento e teste
- Avalie o tempo de processamento

In [126]:
%%time

# Cria um objeto PCA para reduzir a dimensionalidade dos dados, mantendo todas as 561 variáveis
prcomp = PCA(n_components=561).fit(X_train)

# Aplica a transformação PCA nos dados de treino e teste
X_train_pca = prcomp.transform(X_train)
X_test_pca = prcomp.transform(X_test)

# Define o número de componentes principais a serem utilizados
n = 1

# Cria os nomes das colunas para os componentes principais, no formato 'cp1', 'cp2', etc.
coluna = ['cp' + str(x + 1) for x in list(range(n))]

# Cria um DataFrame com os componentes principais do conjunto de treino, utilizando apenas os 'n' primeiros componentes
pc_train = pd.DataFrame(X_train_pca[:, :n], columns=coluna)

# Cria um DataFrame com os componentes principais do conjunto de teste, utilizando apenas os 'n' primeiros componentes
pc_test = pd.DataFrame(X_test_pca[:, :n], columns=coluna)

# Cria e treina um classificador de árvore de decisão com o parâmetro de poda (ccp_alpha)
clf = DecisionTreeClassifier(random_state=4500, ccp_alpha=0.001).fit(pc_train, y_train.values.ravel())

# Calcula e imprime a acurácia do modelo no conjunto de teste
print("Acurácia do modelo de teste: ", round(accuracy_score(y_test, clf.predict(pc_test)), 3))

# Calcula e imprime a acurácia do modelo no conjunto de treino
print("Acurácia do modelo de treino: ", round(accuracy_score(y_train, clf.predict(pc_train)), 3))

Acurácia do modelo de teste:  0.479
Acurácia do modelo de treino:  0.505
CPU times: total: 1.33 s
Wall time: 182 ms


## Testando o número de componentes

Com base no código acima, teste a árvore de classificação com pelo menos as seguintes possibilidades de quantidades de componentes: ```[1, 2, 5, 10, 50]```. Avalie para cada uma delas:

- Acurácia nas bases de treino e teste
- Tempo de processamento


In [127]:
%%time

# Cria um novo classificador de árvore de decisão com um parâmetro de poda (ccp_alpha) e uma semente aleatória para reprodutibilidade
clf = DecisionTreeClassifier(random_state=4500, ccp_alpha=0.001)

# Cria um pipeline que inclui as etapas de escalonamento, PCA e classificação
pipe = Pipeline([
    ('scaler', StandardScaler()),  # Normaliza os dados para que todas as variáveis tenham a mesma escala
    ('pca', PCA()),  # Reduz a dimensionalidade dos dados usando PCA
    ('clf', clf)  # Modelo de classificação, neste caso, uma árvore de decisão
])

# Define os parâmetros para o GridSearchCV
grid_params = {
    'pca__n_components': [1, 2, 5, 10, 50]  # Testa diferentes números de componentes principais
}

# Realiza a busca de hiperparâmetros com validação cruzada
grid = GridSearchCV(pipe, grid_params, cv=15, scoring='accuracy', verbose=1)  # Validação cruzada com 15 folds
grid.fit(X_train, y_train)  # Treina o pipeline com os dados de treino

# Faz previsões no conjunto de teste
y_pred = grid.predict(X_test)

Fitting 15 folds for each of 5 candidates, totalling 75 fits
CPU times: total: 2min 16s
Wall time: 20 s


In [128]:
# Exibe o melhor número de componentes principais encontrado pelo GridSearchCV
print("Melhor número de componentes principais:", grid.best_params_)

# Calcula e imprime a acurácia do modelo no conjunto de teste
# Utiliza as previsões feitas pelo modelo treinado no conjunto de teste (y_pred)
print("Acurácia do modelo de teste: ", round(accuracy_score(y_test, y_pred), 3))

# Calcula e imprime a acurácia do modelo no conjunto de treino
# Utiliza as previsões feitas pelo modelo treinado no conjunto de treino (X_train)
print("Acurácia do modelo de treino: ", round(accuracy_score(y_train, grid.predict(X_train)), 3))

Melhor número de componentes principais: {'pca__n_components': 50}
Acurácia do modelo de teste:  0.831
Acurácia do modelo de treino:  0.91


In [129]:
# Cria um DataFrame a partir dos resultados do GridSearchCV
# Este DataFrame contém informações detalhadas sobre o desempenho do modelo para cada configuração de hiperparâmetros testada
resultados_acuracia = pd.DataFrame(grid.cv_results_)

# Exibe o DataFrame com os resultados, permitindo visualizar métricas como tempo de ajuste, acurácia média, desvio padrão, etc.
resultados_acuracia

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_pca__n_components,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,...,split8_test_score,split9_test_score,split10_test_score,split11_test_score,split12_test_score,split13_test_score,split14_test_score,mean_test_score,std_test_score,rank_test_score
0,0.139496,0.007374,0.006844,0.001148,1,{'pca__n_components': 1},0.445293,0.418367,0.479592,0.474490,...,0.454082,0.484694,0.477041,0.454082,0.482143,0.497449,0.479592,0.471693,0.019913,5
1,0.144136,0.010929,0.006923,0.001083,2,{'pca__n_components': 2},0.582697,0.563776,0.579082,0.561224,...,0.556122,0.602041,0.584184,0.528061,0.594388,0.563776,0.568878,0.576432,0.020733,4
2,0.172441,0.006819,0.006825,0.000900,5,{'pca__n_components': 5},0.829517,0.803571,0.813776,0.811224,...,0.818878,0.823980,0.785714,0.790816,0.811224,0.770408,0.762755,0.797818,0.020463,3
3,0.218012,0.018597,0.007460,0.001003,10,{'pca__n_components': 10},0.844784,0.831633,0.821429,0.849490,...,0.831633,0.836735,0.806122,0.834184,0.841837,0.821429,0.801020,0.826047,0.014229,2
4,0.576791,0.064486,0.007784,0.001539,50,{'pca__n_components': 50},0.875318,0.839286,0.806122,0.834184,...,0.836735,0.841837,0.836735,0.813776,0.882653,0.831633,0.808673,0.835395,0.020510,1


## Conclua

- O que aconteceu com a acurácia?
- O que aconteceu com o tempo de processamento?

### Resposta

* A acurácia demonstrou capacidade de se equiparar a árvore de dados tradicional, se distanciando apenas por 2% e utilizando apenas 10% das variáveis do conjunto de dados originais

* Por termos processado múltiplas árvores através do GridSearchCV afim de encontrar o número de variáveis que demonstrassem o melhor resultado, o tempo de execução do modelo com PCA excedeu o tempo utilizado pelo modelo tradicional de árvore de classificação. É necessário considerar que não foi realizado nenhum Cross Validation na árvore de classificação tradicional